# Hyperparameter Tuning Example

## Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

## Load Data

In [2]:
df = sns.load_dataset('tips')
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [3]:
df.shape

(244, 7)

## Feature Engineering

Defineix els passos de preprocessament per als models que ho necessiten.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [5]:
onehot_pipeline = Pipeline([
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')),
])

In [6]:
onehot_features = ['sex', 'smoker', 'day', 'time']

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', onehot_pipeline, onehot_features),
    ],
    remainder='passthrough',
)

## Train / Test

In [7]:
X = df.drop('tip', axis=1)
y = df['tip']

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

## Cross-validation

In [8]:
# Declare KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

## Decision Tree

If this was not a pipeline, there would be no need to set `'dt__'` before hyperparameter names.

In [9]:
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('dt', DecisionTreeRegressor())
])

param_dist = {
    'dt__max_depth': [3, 10, 50, 100, None],
    'dt__min_samples_split': [5, 8, 10, 15, 20],
    'dt__min_samples_leaf': [2, 5, 10, 15, 20],
    'dt__criterion': ['squared_error', 'absolute_error']
}

dt_gs = GridSearchCV(
    dt_pipeline,
    param_grid=param_dist,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    cv=kf,
    n_jobs=-1
)
dt_gs.fit(X_train, y_train)

print('Best GridSearchCV parameters: ', dt_gs.best_params_)

Best GridSearchCV parameters:  {'dt__criterion': 'squared_error', 'dt__max_depth': 3, 'dt__min_samples_leaf': 2, 'dt__min_samples_split': 8}


In [19]:
dt_gs.best_index_

np.int64(1)

In [21]:
dt_gs.cv_results_['mean_test_score'][dt_gs.best_index_]

np.float64(-0.7917103650563679)

In [ ]:
print('CV Train MAE:', -dt_gs.cv_results_['mean_train_score'][dt_gs.best_index_].round(2))
print('CV Validation MAE:', -dt_gs.cv_results_['mean_test_score'][dt_gs.best_index_].round(2))

CV Train MAE: 0.68
CV Validation MAE: 0.79


### Randomized Search

With a Decistion Tree.

As we're performing a randomized search, we can try wider ranges of hyperparameter values.

#### Round 1

In [ ]:
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('dt', DecisionTreeRegressor())
])

param_dist = {
    'dt__max_depth': list(range(2, 100)) + [None],
    'dt__min_samples_split': range(2, 20),
    'dt__min_samples_leaf': range(2, 20),
    'dt__criterion': ['squared_error', 'absolute_error']
}

dt_rs = RandomizedSearchCV(
    dt_pipeline,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    cv=kf,
    random_state=42,
    n_jobs=-1
)
dt_rs.fit(X_train, y_train)

print('Best RandomizedSearchCV parameters: ', dt_rs.best_params_)

Best RandomizedSearchCV parameters:  {'dt__min_samples_split': 15, 'dt__min_samples_leaf': 13, 'dt__max_depth': 48, 'dt__criterion': 'absolute_error'}


In [28]:
print('CV Train MAE:', -dt_rs.cv_results_['mean_train_score'][dt_rs.best_index_].round(2))
print('CV Validation MAE:', -dt_rs.cv_results_['mean_test_score'][dt_rs.best_index_].round(2))

CV Train MAE: 0.64
CV Validation MAE: 0.77


In [30]:
# Get the indices of the smallest MAE values
best_indices = np.argsort(-dt_rs.cv_results_['mean_test_score'])[:5]

# Show best MAEs and their corresponding hyperparams.
for i in best_indices:
    print('MAE:', -dt_rs.cv_results_['mean_test_score'][i].round(2))
    print('Hyperparams:', dt_rs.cv_results_['params'][i])
    print()

MAE: 0.77
Hyperparams: {'dt__min_samples_split': 15, 'dt__min_samples_leaf': 13, 'dt__max_depth': 48, 'dt__criterion': 'absolute_error'}

MAE: 0.77
Hyperparams: {'dt__min_samples_split': 4, 'dt__min_samples_leaf': 13, 'dt__max_depth': 90, 'dt__criterion': 'absolute_error'}

MAE: 0.78
Hyperparams: {'dt__min_samples_split': 4, 'dt__min_samples_leaf': 12, 'dt__max_depth': 85, 'dt__criterion': 'absolute_error'}

MAE: 0.78
Hyperparams: {'dt__min_samples_split': 8, 'dt__min_samples_leaf': 12, 'dt__max_depth': 78, 'dt__criterion': 'absolute_error'}

MAE: 0.8
Hyperparams: {'dt__min_samples_split': 3, 'dt__min_samples_leaf': 15, 'dt__max_depth': 19, 'dt__criterion': 'absolute_error'}



#### Round 2

In [ ]:
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('dt', DecisionTreeRegressor(criterion='absolute_error'))
])

param_dist = {
    'dt__max_depth': range(50, 91),
    'dt__min_samples_split': range(3, 16),
    'dt__min_samples_leaf': range(10, 16),
}

dt_rs = RandomizedSearchCV(
    dt_pipeline,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    cv=kf,
    n_jobs=-1
)
dt_rs.fit(X_train, y_train)

print('Best RandomizedSearchCV parameters: ', dt_rs.best_params_)

Best RandomizedSearchCV parameters:  {'dt__min_samples_split': np.int64(6), 'dt__min_samples_leaf': 13, 'dt__max_depth': 61}


In [32]:
print('CV Train MAE:', -dt_rs.cv_results_['mean_train_score'][dt_rs.best_index_].round(2))
print('CV Validation MAE:', -dt_rs.cv_results_['mean_test_score'][dt_rs.best_index_].round(2))

CV Train MAE: 0.64
CV Validation MAE: 0.77
